# 16a · Semantic caches need exact context

## The engineering question
When is a similar question safe to answer from a cache, and when must the cache miss?

**Learning outcomes:** separate similarity from cache eligibility; include tenant, permission/source revisions, conversation state, and answer-model revision; enforce expiration and invalidation in Qdrant; measure false hits on negation and product changes.

The cached answer is an explicit fixture, not a generated response. Qdrant stores the query embedding and exact scope hash. Scope and expiry filters run before similarity search. A hit still requires semantic equivalence for the application; a cosine threshold cannot prove that. The threshold 0.9 is fixed before this tiny diagnostic set, not optimized on it.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("16a", "B", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
from uuid import uuid4
from vector_masterclass.semantic_cache import SemanticCache, CacheScope

cache = SemanticCache("vmc_16a_" + uuid4().hex[:8])
scope = CacheScope(
    tenant="red",
    authorization_revision="1",
    source_id="atlas-warranty",
    source_revision="1",
    conversation_hash="product=Atlas",
    answer_model_revision="fixture-v1",
)
tests = [
    ("exact", "How long is the Atlas warranty?", True),
    ("paraphrase", "What is the warranty duration for Atlas?", True),
    ("wrong product", "How long is the Boreal warranty?", False),
    ("negated request", "Does the Atlas warranty not last two years?", False),
    ("different fact", "What is the Atlas waterproof rating?", False),
]
rows = []
try:
    cache.put(tests[0][1], "Fixture answer: two years.", scope, now=100, ttl=30)
    for label, query, safe in tests:
        hit = cache.get(query, scope, now=110, threshold=0.9)
        rows.append(
            {
                "case": label,
                "expected_safe_reuse": safe,
                "cache_hit": hit is not None,
                "false_hit": hit is not None and not safe,
                "score": None if hit is None else hit["score"],
                "scope_constraint": False,
            }
        )
    for field in CacheScope.model_fields:
        changed = scope.model_copy(update={field: "changed"})
        hit = cache.get(tests[0][1], changed, now=110, threshold=0.9)
        rows.append(
            {
                "case": "changed " + field,
                "expected_safe_reuse": False,
                "cache_hit": hit is not None,
                "false_hit": hit is not None,
                "score": None,
                "scope_constraint": True,
            }
        )
    expired = cache.get(tests[0][1], scope, now=130, threshold=0.9)
    cache.invalidate_source("atlas-warranty")
    deleted = cache.get(tests[0][1], scope, now=110, threshold=0.9)
finally:
    cache.close(drop=True)
results = lesson.table(pd.DataFrame(rows))
display(results)
fig, ax = plt.subplots(figsize=(10, 5))
values = results.cache_hit.astype(int)
ax.barh(
    results["case"],
    values,
    color=["#dc2626" if false else "#2563eb" for false in results.false_hit],
)
ax.set(
    xlabel="Cache hit (1) or miss (0)",
    xticks=[0, 1],
    title="Context changes must miss, regardless of query similarity",
)
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Interpretation and exercise
Exact scope protects eligibility, not meaning. A negated question may remain close enough to the original for a false hit; if no false hit appears at 0.9, lower the threshold on a separate development set and inspect the first failure. A missed paraphrase costs latency; a false answer can change the user's decision. Choose asymmetric error costs and consider exact intent/entity guards or a verifier.

**Exercise:** warm a cache, revoke access, and show why the authoritative authorization revision must change before the next lookup. Then change only the conversation's product identity. Explain why TTL alone cannot implement immediate revocation or source deletion. The notebook uses an explicit synthetic clock for deterministic expiry tests; a deployed service needs one trusted clock and a skew policy.

Invalidation removes all entries for the source across scopes. Request callers must not manufacture their own scope; the service constructs it from authoritative state. A distributed cache also needs deletion propagation, observability, and memory/eviction limits. Cached values are prior outputs or fixtures, never evidence of a new model call.

In [ ]:
manifest = lesson.finish(
    {
        "exact_query_hits": bool(results.iloc[0].cache_hit),
        "all_context_changes_miss": bool(
            (~results.loc[results.scope_constraint, "cache_hit"]).all()
        ),
        "expiry_boundary_excluded": expired is None,
        "source_invalidation_excluded": deleted is None,
        "negative_semantic_cases_measured": int(
            (~results.expected_safe_reuse & ~results.scope_constraint).sum()
        )
        == 3,
    },
    observations={
        "clock": "synthetic seconds 100..130",
        "threshold": 0.9,
        "answers": "explicit fixtures; no generation claim",
        "semantic_false_hits": int(results.false_hit.sum()),
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.